# Notebook 06 — Robust Optimisation under Uncertainty (CVaR MILP)

This notebook upgrades the v1.0 deterministic squad selection problem into an uncertainty-aware decision system.

We model per-player value under uncertainty via scenario generation and solve a **CVaR (Conditional Value-at-Risk)** optimisation that remains **MILP-compatible** and solvable with **`scipy.optimize.milp` (HiGHS)**.

**Key deliverables**
- Scenario-based uncertainty model for talent/value
- CVaR linear reformulation (MILP)
- Deterministic vs CVaR-robust squad comparison
- Sensitivity: **α** (tail confidence) and **budget** (constraint tightness)
- Conclusions for decision-making

> Designed to run top-to-bottom after `Restart Kernel` → `Run All`.

## 0) Imports and global settings

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import duckdb

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy import sparse

import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## 1) Configuration and data access

In [ ]:
# --- Experiment configuration (edit here) ---
SQUAD_SIZE: int = 18
BUDGET_EUR: int = 180_000_000
LAMBDA_RISK: float = 0.5

# CVaR configuration
ALPHA: float = 0.80
ENGINE: str = "mc"      # "regime" or "mc"
S_SCEN: int = 150
SEED: int = 42

# Tail severity (MC engine)
Q_BAD: float = 0.25
BAD_SHIFT: float = -1.75

# Universe filters
MIN_MINUTES: int = 900

# Broad position groups (from risk_adjusted_universe_v2)
POS_GROUPS: Dict[str, List[str]] = {
    "DF": ["Defender"],
    "MF": ["Midfield"],
    "FW": ["Attack"],
}
POS_L: Dict[str, int] = {"DF": 6, "MF": 6, "FW": 6}
POS_U: Dict[str, int] = {"DF": 8, "MF": 8, "FW": 8}

assert sum(POS_L.values()) <= SQUAD_SIZE <= sum(POS_U.values()), "Quota sums incompatible with SQUAD_SIZE."

In [ ]:
def find_db_path(db_name: str = "scouting.duckdb") -> Path:
    """Find db/scouting.duckdb robustly from repo_root/ or notebooks/."""
    cwd = Path.cwd()
    candidates = [cwd / "db" / db_name, cwd.parent / "db" / db_name]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"Could not find {db_name}. Tried: {candidates}")


DB_PATH = find_db_path()
print("Using DB at:", DB_PATH)

# Close previous connection if it exists
try:
    con.close()
except:
    pass

con = duckdb.connect(str(DB_PATH))

### 1.1 Load candidate universe

In [ ]:
SQL = '''
SELECT
    player_id,
    player_name,
    season,
    age,
    season_minutes,
    position                 AS pos_primary,
    talent_score             AS mu_talent,
    risk_score               AS risk,
    market_value_in_eur      AS cost_proxy_eur,
    age_distance,
    usage_instability
FROM risk_adjusted_universe_v2
'''
df = con.execute(SQL).df()

df = df[df["season_minutes"] >= MIN_MINUTES].copy()
df = df.dropna(subset=["mu_talent","risk","cost_proxy_eur","pos_primary","age_distance","usage_instability"])
df.reset_index(drop=True, inplace=True)

print("Players after filters:", len(df))
print("Positions:", sorted(df["pos_primary"].unique()))
display(df.head(3))

## 2) Deterministic value and uncertainty scale σ

In [ ]:
def build_sigma(minutes: np.ndarray, vol: np.ndarray, age_dist: np.ndarray,
                w_minutes: float = 0.35, w_vol: float = 0.45, w_age: float = 0.20,
                floor: float = 0.15, cap: float = 2.50) -> np.ndarray:
    """Interpretable uncertainty proxy σ_i."""
    minutes = np.asarray(minutes, float)
    vol = np.asarray(vol, float)
    age_dist = np.asarray(age_dist, float)

    m = np.log1p(minutes)
    m_z = (m - m.mean()) / (m.std() + 1e-9)
    u_minutes = -m_z

    v_z = (vol - vol.mean()) / (vol.std() + 1e-9)
    a_z = (age_dist - age_dist.mean()) / (age_dist.std() + 1e-9)

    sigma = w_minutes * u_minutes + w_vol * v_z + w_age * a_z
    sigma = (sigma - sigma.min()) / (sigma.max() - sigma.min() + 1e-9)
    sigma = floor + sigma * (cap - floor)
    return sigma

mu = df["mu_talent"].to_numpy(float)
risk = df["risk"].to_numpy(float)
tco = df["cost_proxy_eur"].to_numpy(float)

minutes = df["season_minutes"].to_numpy(float)
vol = df["usage_instability"].to_numpy(float)
age_dist = df["age_distance"].to_numpy(float)

value_mean = mu - LAMBDA_RISK * risk
sigma = build_sigma(minutes, vol, age_dist)
df["sigma"] = sigma

print("value_mean summary:\n", pd.Series(value_mean).describe())
print("sigma summary:\n", pd.Series(sigma).describe())

## 3) Scenario generation

In [ ]:
@dataclass
class ScenarioSet:
    V: np.ndarray
    p: np.ndarray
    alpha: float
    scenario_names: Optional[List[str]] = None


def make_regime_scenarios(mu, sigma, risk, lam,
                          p=(0.70, 0.20, 0.10),
                          shock_levels=(0.0, -0.75, -1.75),
                          alpha=0.80) -> ScenarioSet:
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    risk = np.asarray(risk, float)

    p = np.asarray(p, float); p = p / p.sum()
    shocks = np.asarray(shock_levels, float)
    if len(p) != len(shocks):
        raise ValueError("p and shock_levels must have same length")

    T = mu[None, :] + sigma[None, :] * shocks[:, None]
    V = T - lam * risk[None, :]
    names = [f"regime_{k}_shock_{shocks[k]:+.2f}" for k in range(len(shocks))]
    return ScenarioSet(V=V, p=p, alpha=float(alpha), scenario_names=names)


def make_factor_mc_scenarios(mu, sigma, risk, lam,
                             S=150, alpha=0.80, seed=42,
                             q_bad=0.25, bad_shift=-1.75,
                             eps_scale=0.35,
                             exposure_clip=(0.5, 1.5)) -> ScenarioSet:
    rng = np.random.default_rng(seed)
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    risk = np.asarray(risk, float)
    n = len(mu)

    p = np.full(S, 1.0 / S, dtype=float)

    sigma_z = (sigma - sigma.mean()) / (sigma.std() + 1e-9)
    a = 1.0 + 0.25 * sigma_z
    a = np.clip(a, exposure_clip[0], exposure_clip[1])

    bad = rng.random(S) < q_bad
    f = rng.standard_normal(S)
    f[bad] = rng.standard_normal(bad.sum()) + bad_shift

    eps = rng.standard_normal((S, n)) * eps_scale
    Xi = f[:, None] * a[None, :] + eps

    T = mu[None, :] + sigma[None, :] * Xi
    V = T - lam * risk[None, :]
    return ScenarioSet(V=V, p=p, alpha=float(alpha))


print("Generating scenarios...")
if ENGINE == "regime":
    scen = make_regime_scenarios(mu, sigma, risk, LAMBDA_RISK, alpha=ALPHA)
else:
    scen = make_factor_mc_scenarios(
        mu, sigma, risk, LAMBDA_RISK,
        S=S_SCEN, alpha=ALPHA, seed=SEED,
        q_bad=Q_BAD, bad_shift=BAD_SHIFT
    )

V_scen = scen.V
p = scen.p
print("V_scen shape:", V_scen.shape, "sum(p)=", p.sum())

display(pd.DataFrame({
    "player_name": df["player_name"].head(5),
    "pos": df["pos_primary"].head(5),
    "mu": mu[:5],
    "sigma": sigma[:5],
    "v_s0": V_scen[0, :5],
}))

## 4) Core constraints

In [ ]:
def build_position_matrix(pos, pos_groups, pos_L, pos_U):
    pos = np.asarray(pos)
    group_names = list(pos_groups.keys())
    G = len(group_names)
    n = len(pos)

    A = np.zeros((G, n), dtype=float)
    for g, name in enumerate(group_names):
        A[g, :] = np.isin(pos, pos_groups[name]).astype(float)

    L = np.array([float(pos_L[name]) for name in group_names], dtype=float)
    U = np.array([float(pos_U[name]) for name in group_names], dtype=float)
    return A, L, U, group_names

A_pos, L, U, group_names = build_position_matrix(df["pos_primary"].to_numpy(), POS_GROUPS, POS_L, POS_U)
print("Group counts:", {name: int(A_pos[g,:].sum()) for g, name in enumerate(group_names)})
print("Sum L / U / N:", L.sum(), U.sum(), SQUAD_SIZE)

In [ ]:
def build_core_constraints_x(n, tco, budget, A_pos, L, U, squad_size):
    A_ub = sparse.vstack([
        sparse.csr_matrix(tco.reshape(1, -1)),
        sparse.csr_matrix(A_pos),
        sparse.csr_matrix(-A_pos),
    ], format="csr")
    b_ub = np.concatenate([np.array([budget], float), U.astype(float), (-L).astype(float)])

    A_eq = sparse.csr_matrix(np.ones((1, n), float))
    b_eq = np.array([float(squad_size)], float)
    return A_ub, b_ub, A_eq, b_eq

n = len(df)
A_ub_x, b_ub_x, A_eq_x, b_eq_x = build_core_constraints_x(n, tco, BUDGET_EUR, A_pos, L, U, SQUAD_SIZE)
print("Core constraints built:", A_ub_x.shape, A_eq_x.shape)

## 5) MILP solvers + CVaR MILP

In [ ]:
def solve_milp(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=None, integrality=None):
    constraints = []
    if A_ub is not None:
        constraints.append(LinearConstraint(A_ub, -np.inf, b_ub))
    if A_eq is not None:
        constraints.append(LinearConstraint(A_eq, b_eq, b_eq))
    if bounds is None:
        bounds = Bounds(-np.inf, np.inf)
    return milp(c=c, constraints=constraints, bounds=bounds, integrality=integrality)


def solve_deterministic_milp(value_mean, A_ub_x, b_ub_x, A_eq_x, b_eq_x):
    c = (-value_mean).astype(float)  # maximize => minimize negative
    bounds = Bounds(np.zeros_like(c), np.ones_like(c))
    integrality = np.ones_like(c, dtype=int)
    res = solve_milp(c, A_ub_x, b_ub_x, A_eq_x, b_eq_x, bounds=bounds, integrality=integrality)
    if not res.success:
        raise RuntimeError(res.message)
    return res.x


def build_cvar_value_milp(V_scen, p, alpha, A_ub_x, b_ub_x, A_eq_x, b_eq_x):
    # Variables: z = [x (n), eta (1), u (S)]
    # Maximise: eta - 1/(1-alpha) * sum p_s u_s
    # Minimise: -eta + 1/(1-alpha) * sum p_s u_s
    S, n = V_scen.shape
    c = np.concatenate([np.zeros(n), np.array([-1.0]), (p / (1.0 - alpha))])

    # u_s >= eta - V_s(x)  <=>  -Vx + eta - u <= 0
    A_cvar = sparse.hstack([
        sparse.csr_matrix(-V_scen),
        sparse.csr_matrix(np.ones((S, 1))),
        sparse.identity(S, format="csr") * (-1.0),
    ], format="csr")
    b_cvar = np.zeros(S)

    A_ub_lift = sparse.hstack([
        A_ub_x,
        sparse.csr_matrix((A_ub_x.shape[0], 1)),
        sparse.csr_matrix((A_ub_x.shape[0], S)),
    ], format="csr")

    A_eq_lift = sparse.hstack([
        A_eq_x,
        sparse.csr_matrix((A_eq_x.shape[0], 1)),
        sparse.csr_matrix((A_eq_x.shape[0], S)),
    ], format="csr")

    A_ub = sparse.vstack([A_cvar, A_ub_lift], format="csr")
    b_ub = np.concatenate([b_cvar, b_ub_x])

    A_eq = A_eq_lift
    b_eq = b_eq_x

    lb = np.concatenate([np.zeros(n), np.array([-np.inf]), np.zeros(S)])
    ub = np.concatenate([np.ones(n),  np.array([ np.inf]), np.full(S, np.inf)])
    integrality = np.concatenate([np.ones(n, int), np.zeros(1 + S, int)])

    return c, A_ub, b_ub, A_eq, b_eq, (lb, ub), integrality


def solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality):
    res = solve_milp(c, A_ub, b_ub, A_eq, b_eq, bounds=Bounds(bounds[0], bounds[1]), integrality=integrality)
    if not res.success:
        raise RuntimeError(res.message)
    return res.x

## 6) Solve and compare

In [ ]:
def cvar_left(values, alpha):
    v = np.sort(values)
    k = max(1, int(np.ceil((1.0 - alpha) * len(v))))
    return float(v[:k].mean())

x_det = solve_deterministic_milp(value_mean, A_ub_x, b_ub_x, A_eq_x, b_eq_x)
x_det_bin = (x_det > 0.5).astype(int)

c, A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_cvar_value_milp(V_scen, p, ALPHA, A_ub_x, b_ub_x, A_eq_x, b_eq_x)
z = solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality)
x_rob_bin = (z[:n] > 0.5).astype(int)

vals_det = V_scen @ x_det_bin
vals_rob = V_scen @ x_rob_bin

print("Selected det/rob:", int(x_det_bin.sum()), int(x_rob_bin.sum()))
print("Mean det/rob:", float(vals_det.mean()), float(vals_rob.mean()))
print("CVaR det/rob:", cvar_left(vals_det, ALPHA), cvar_left(vals_rob, ALPHA))
print("P10 det/rob:", float(np.quantile(vals_det, 0.10)), float(np.quantile(vals_rob, 0.10)))

## 7) α sweep and budget sweep

In [ ]:
def run_alpha_sweep(alpha_grid):
    rows = []
    sols = {}
    for a in alpha_grid:
        c, A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_cvar_value_milp(V_scen, p, float(a), A_ub_x, b_ub_x, A_eq_x, b_eq_x)
        z = solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality)
        x_bin = (z[:n] > 0.5).astype(int)
        sols[a] = x_bin
        vals = V_scen @ x_bin
        rows.append({
            "alpha": float(a),
            "scenario_mean": float(vals.mean()),
            "scenario_cvar": cvar_left(vals, float(a)),
            "p10": float(np.quantile(vals, 0.10)),
            "total_cost": float(tco @ x_bin),
            "avg_sigma_per_player": float(df.loc[x_bin.astype(bool), "sigma"].mean()),
        })
    frontier = pd.DataFrame(rows).sort_values("alpha").reset_index(drop=True)
    base = sols[alpha_grid[0]]
    diffs = {a: int(np.sum(base != sols[a])) for a in alpha_grid[1:]}
    return frontier, diffs

frontier_alpha, alpha_diffs = run_alpha_sweep([0.70, 0.75, 0.80, 0.85, 0.90])
display(frontier_alpha)
print("Hamming diffs vs first alpha:", alpha_diffs)


def solve_both_for_budget(budget_eur):
    A_ub_x_b, b_ub_x_b, A_eq_x_b, b_eq_x_b = build_core_constraints_x(n, tco, budget_eur, A_pos, L, U, SQUAD_SIZE)

    x_det = solve_deterministic_milp(value_mean, A_ub_x_b, b_ub_x_b, A_eq_x_b, b_eq_x_b)
    x_det_bin = (x_det > 0.5).astype(int)

    c, A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_cvar_value_milp(V_scen, p, ALPHA, A_ub_x_b, b_ub_x_b, A_eq_x_b, b_eq_x_b)
    z = solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality)
    x_rob_bin = (z[:n] > 0.5).astype(int)

    vals_det = V_scen @ x_det_bin
    vals_rob = V_scen @ x_rob_bin

    return {
        "budget_eur": float(budget_eur),
        "det_mean": float(vals_det.mean()),
        "rob_mean": float(vals_rob.mean()),
        "det_cvar": cvar_left(vals_det, ALPHA),
        "rob_cvar": cvar_left(vals_rob, ALPHA),
        "det_p10": float(np.quantile(vals_det, 0.10)),
        "rob_p10": float(np.quantile(vals_rob, 0.10)),
        "det_avg_sigma": float(df.loc[x_det_bin.astype(bool), "sigma"].mean()),
        "rob_avg_sigma": float(df.loc[x_rob_bin.astype(bool), "sigma"].mean()),
        "hamming_diff": int(np.sum(x_det_bin != x_rob_bin)),
    }

budget_sweep = pd.DataFrame([solve_both_for_budget(b) for b in [170e6, 180e6, 190e6, 210e6]]).sort_values("budget_eur")
display(budget_sweep)

In [ ]:
premium = budget_sweep.copy()

premium["delta_mean"] = premium["rob_mean"] - premium["det_mean"]
premium["delta_cvar"] = premium["rob_cvar"] - premium["det_cvar"]
premium["delta_p10"] = premium["rob_p10"] - premium["det_p10"]

display(
    premium[[
        "budget_eur",
        "delta_mean",
        "delta_cvar",
        "delta_p10",
        "hamming_diff"
    ]]
)

## Downside Performance vs Budget

In [ ]:
plt.figure()

plt.plot(budget_sweep["budget_eur"]/1e6,
         budget_sweep["det_p10"],
         marker="o",
         label="Deterministic")

plt.plot(budget_sweep["budget_eur"]/1e6,
         budget_sweep["rob_p10"],
         marker="o",
         label="Robust (CVaR)")

plt.xlabel("Budget (M€)")
plt.ylabel("P10 scenario value")
plt.title("Downside performance vs budget")
plt.legend()

plt.show()

## 8) Conclusions

### Summary

- CVaR optimisation **raises downside performance** (CVaR / P10) and typically selects players with **lower σ**.
- The robust-vs-deterministic gap is **budget-dependent**: it becomes more pronounced under tighter budgets.
- α-sweeps can be invariant in practice—this is a *stability* result, not a failure.

### Practical takeaway for recruitment

Use the deterministic model as a baseline shortlist, then run CVaR as a risk-control layer when:
- budgets are tight,
- positional quotas are binding,
- player uncertainty differs materially across the candidate universe.

Track the **robust premium**: ΔMean vs ΔCVaR / ΔP10 to quantify the price of downside protection.

### Robust optimisation insights

The CVaR optimisation framework materially improves downside performance of the squad portfolio.

Across all tested budgets:

- deterministic solutions achieve higher expected value

- robust solutions significantly improve tail performance (CVaR and P10)

- robust squads systematically exhibit lower average uncertainty (σ)

The structural differences between squads (Hamming distance between 16 and 20 players) show that downside-aware recruitment produces fundamentally different squad compositions rather than minor adjustments.

This suggests that deterministic scouting pipelines may systematically overexpose clubs to volatility risk.

### Model limitations

This framework models uncertainty through simulated performance shocks.
Transfer fees and wages are approximated through market value proxies,
and player interactions (chemistry, tactical fit) are not explicitly modelled.

Future work could incorporate:
- correlated positional shocks
- transfer fee vs wage cost structures
- squad age balance constraints

## Reproducibility

Random seeds and scenario engines are fully configurable.
All results can be reproduced by re-running the notebook from top to bottom.

Environment:
- Python
- DuckDB
- SciPy MILP (HiGHS)